In [7]:
from pathlib import Path
from graph_ir.data.collection import Collection
from graph_ir.models.graphical_setbased import GSBModel
import numpy as np

# Locate the repo root regardless of where Jupyter's working directory is
# (it is the notebook's folder by default, not the repo root).
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root (no pyproject.toml found)")

REPO_ROOT = find_repo_root(Path.cwd())
cf_path = REPO_ROOT / "collections" / "cf"
print(f"Loading CF from {cf_path}...")

Loading CF from /home/nikitas/PycharmProjects/Graph_ir/collections/cf...


In [8]:
from functools import partial
from graph_ir.data.preprocessing.text import preprocess_text

# New experiment: lowercase, split punctuation, remove English stopwords.
# Keep numbers; do not stem. The model applies these same rules to queries.
collection = Collection(cf_path)
preprocessor = partial(preprocess_text, stop_words=set(collection.stopwords))

example = "THE CF patients, patients! IL-6"
print(f"Before: {example}")
print(f"After:  {preprocessor(example)}")

Before: THE CF patients, patients! IL-6
After:  ['cf', 'patients', 'patients', 'il', '6']


In [9]:
# Preprocess document terms before building the inverted index.
collection.create(preprocessor=preprocessor).load_queries().load_qrels()

print("Collection stats:")
print(f"  Documents: {collection.num_docs}")
print(f"  Queries: {len(collection.queries)}")
print(f"  Vocabulary: {len(collection.inverted_index)} terms")
print(f"  Relevance judgments: {len(collection.qrels)} queries with qrels")

sample_qid = next(iter(collection.queries))
sample_query = collection.queries[sample_qid]
print(f"Sample query {sample_qid}: {sample_query}")
print(f"Query tokens: {preprocessor(sample_query)}")
print(f"First document tokens: {collection.docs[0].terms[:15]}")

Collection stats:
  Documents: 1239
  Queries: 100
  Vocabulary: 9987 terms
  Relevance judgments: 100 queries with qrels
Sample query 1: What are the effects of calcium on the physical properties of mucus from CF patients?
Query tokens: ['effects', 'calcium', 'physical', 'properties', 'mucus', 'cf', 'patients']
First document tokens: ['pseudomonas', 'aeruginosa', 'infection', 'cystic', 'fibrosis', 'occurrence', 'precipitating', 'antibodies', 'pseudomonas', 'aeruginosa', 'relation', 'concentration', 'sixteen', 'serum', 'proteins']


In [10]:
model = GSBModel(collection)
print(f"Model: {model.get_model()}")


Model: GSBModel


In [11]:
import time

# Fit all loaded queries. Their preprocessing comes from the collection.
print("Fitting model (Apriori termset mining)...")
start = time.perf_counter()
model.fit(min_freq=1)

elapsed = time.perf_counter() - start
print(f"Fit complete in {elapsed:.2f}s")
print(f"Queries fitted: {len(model._query_ids)}")
print(f"Total termsets mined: {sum(len(ts) for ts in model._docVectors)}")

Fitting model (Apriori termset mining)...
Fit complete in 0.12s
Queries fitted: 100
Total termsets mined: 8999


In [12]:
print("Evaluating...")
precisions, recalls = model.evaluate()

mean_prec = np.mean(precisions)
mean_rec = np.mean(recalls)

# The existing helpers average precision/recall at relevant hits.
# These are not standard MAP or recall@k; their formulas are unchanged.
print(f"  Mean precision at relevant hits: {mean_prec:.3f}")
print(f"  Mean recall at relevant hits: {mean_rec:.3f}")

Evaluating...
  Mean precision at relevant hits: 0.178
  Mean recall at relevant hits: 0.523
